# Popolamento dell'Ontologia — Beni Confiscati alla Mafia

Questo notebook legge il dataset processato da `cleaning.ipynb` e costruisce un **grafo RDF** secondo l'ontologia definita in `ontology.ttl`.

Il grafo viene arricchito con **interlinking verso Wikidata**: ogni Comune viene collegato al proprio item Wikidata tramite `owl:sameAs`, usando il codice ISTAT (`wdt:P635`) come chiave di collegamento. L'output finale è il file `ontologia-beni-confiscati.ttl`.

In [ ]:
import pandas as pd
import numpy as np
import re
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('utils.py')))
from rdflib import Graph, Literal, Namespace, URIRef
from rdflib.namespace import RDF, RDFS, XSD, OWL

## 1. Caricamento dati

In [ ]:
PATH_CLEAN    = '../dataset/processed/beni-confiscati.csv'
PATH_WIKIDATA = '../dataset/other/wikidata_comuni.csv'

df = pd.read_csv(PATH_CLEAN, low_memory=False)
print(f'Beni caricati: {len(df):,}')
print(f'Colonne: {list(df.columns)}')

## 2. Interlinking: codici ISTAT → Wikidata

In [ ]:
if os.path.exists(PATH_WIKIDATA):
    df_wd = pd.read_csv(PATH_WIKIDATA)
    if 'item' in df_wd.columns and 'wikidata_uri' not in df_wd.columns:
        df_wd = df_wd.rename(columns={'item': 'wikidata_uri'})
    if 'istat' in df_wd.columns and 'codistat' not in df_wd.columns:
        df_wd = df_wd.rename(columns={'istat': 'codistat'})
    df_wd['codistat'] = df_wd['codistat'].astype(str).str.zfill(6)
    istat_to_wd = dict(zip(df_wd['codistat'], df_wd['wikidata_uri']))
    print(f'Mapping caricato: {len(istat_to_wd):,} comuni con URI Wikidata')
else:
    istat_to_wd = {}
    print('File wikidata_comuni.csv non trovato — interlinking disabilitato.')

## 3. Inizializzazione del grafo RDF

In [ ]:
g = Graph()

EX_ONT  = Namespace('http://example.org/ontology/beniconfiscati/')
EX_DATA = Namespace('http://example.org/data/beniconfiscati/')
WD      = Namespace('http://www.wikidata.org/entity/')

g.bind('exOnt',  EX_ONT)
g.bind('exData', EX_DATA)
g.bind('wd',     WD)
g.bind('xsd',    XSD)
g.bind('owl',    OWL)

g.parse('../ontology/ontology.ttl', format='turtle')
print(f'Triple dopo import ontologia: {len(g)}')

## 4. Popolamento: Regioni

In [ ]:
_pat = re.compile(r'[^\w]')

regioni_uniche = df['NomeRegioneValidato'].dropna().unique()

for regione in regioni_uniche:
    nome_uri = _pat.sub('_', regione.lower().strip())
    uri = EX_DATA[f'regione_{nome_uri}']
    g.add((uri, RDF.type,         EX_ONT.Regione))
    g.add((uri, EX_ONT.nomeLabel, Literal(regione, datatype=XSD.string)))

print(f'Regioni aggiunte: {len(regioni_uniche)}')
print(f'Triple totali: {len(g)}')

## 5. Popolamento: Province

In [ ]:
province_uniche = (
    df[['NomeProvinciaValidato', 'NomeRegioneValidato', 'CODISTATPROV']]
    .dropna(subset=['NomeProvinciaValidato'])
    .drop_duplicates(subset=['NomeProvinciaValidato'])
)

for _, row in province_uniche.iterrows():
    nome_prov = row['NomeProvinciaValidato']
    nome_reg  = row['NomeRegioneValidato']

    uri_prov = EX_DATA['provincia_' + _pat.sub('_', nome_prov.lower())]
    g.add((uri_prov, RDF.type,         EX_ONT.Provincia))
    g.add((uri_prov, EX_ONT.nomeLabel, Literal(nome_prov, datatype=XSD.string)))

    if pd.notna(nome_reg):
        uri_reg = EX_DATA['regione_' + _pat.sub('_', nome_reg.lower())]
        g.add((uri_prov, EX_ONT.inRegione, uri_reg))

print(f'Province aggiunte: {len(province_uniche)}')
print(f'Triple totali: {len(g)}')

## 6. Popolamento: Comuni

In [ ]:
comuni_unici = (
    df[['NomeComuneValidato', 'NomeProvinciaValidato', 'CODISTAT',
        'CODISTATPROV', 'lat', 'lon']]
    .dropna(subset=['NomeComuneValidato'])
    .drop_duplicates(subset=['CODISTAT'])
)

n_interlinked = 0

for _, row in comuni_unici.iterrows():
    nome_c    = row['NomeComuneValidato']
    codistat  = str(row['CODISTAT']).zfill(6)
    nome_prov = row['NomeProvinciaValidato']

    uri_comune = EX_DATA[f'comune_{codistat}']
    g.add((uri_comune, RDF.type,           EX_ONT.Comune))
    g.add((uri_comune, EX_ONT.nomeLabel,   Literal(nome_c, datatype=XSD.string)))
    g.add((uri_comune, EX_ONT.codiceISTAT, Literal(codistat, datatype=XSD.string)))

    if pd.notna(row['lat']) and pd.notna(row['lon']):
        g.add((uri_comune, EX_ONT.lat, Literal(float(row['lat']), datatype=XSD.decimal)))
        g.add((uri_comune, EX_ONT.lon, Literal(float(row['lon']), datatype=XSD.decimal)))

    if pd.notna(nome_prov):
        uri_prov = EX_DATA['provincia_' + _pat.sub('_', nome_prov.lower())]
        g.add((uri_comune, EX_ONT.inProvincia, uri_prov))

    if codistat in istat_to_wd:
        wd_uri = URIRef(istat_to_wd[codistat])
        g.add((uri_comune, OWL.sameAs, wd_uri))
        n_interlinked += 1

print(f'Comuni aggiunti: {len(comuni_unici)}')
print(f'Comuni interlinked con Wikidata: {n_interlinked}')
print(f'Triple totali: {len(g)}')

## 7. Popolamento: Beni confiscati

In [ ]:
def slugify(s):
    s = str(s).lower().strip()
    s = _pat.sub('_', s)
    s = re.sub(r'_+', '_', s).strip('_')
    return s

CLASS_MAP = {'immobile': EX_ONT.Immobile, 'azienda': EX_ONT.Azienda}

for _, row in df.iterrows():
    bene_id  = str(row['s_bene']).strip()
    uri_bene = EX_DATA['bene_' + slugify(bene_id)]

    classe = CLASS_MAP.get(str(row['tipo']), EX_ONT.BeneConfiscato)
    g.add((uri_bene, RDF.type, classe))
    g.add((uri_bene, EX_ONT.idBene, Literal(bene_id, datatype=XSD.string)))
    g.add((uri_bene, EX_ONT.stato,  Literal(str(row['stato']), datatype=XSD.string)))

    if pd.notna(row['categoria']):
        g.add((uri_bene, EX_ONT.categoria, Literal(str(row['categoria']), datatype=XSD.string)))
    if pd.notna(row['sottocategoria']):
        g.add((uri_bene, EX_ONT.sottocategoria, Literal(str(row['sottocategoria']), datatype=XSD.string)))

    codistat = str(row['CODISTAT']).zfill(6)
    if codistat != '000000':
        g.add((uri_bene, EX_ONT.situatoIn, EX_DATA[f'comune_{codistat}']))

    if row['stato'] == 'destinato':
        if pd.notna(row['destinatario']):
            g.add((uri_bene, EX_ONT.assegnatoA, Literal(str(row['destinatario']), datatype=XSD.string)))
        if pd.notna(row['fine']):
            g.add((uri_bene, EX_ONT.fineUso, Literal(str(row['fine']), datatype=XSD.string)))
        if pd.notna(row['decreto_anno']):
            g.add((uri_bene, EX_ONT.annoDecreto, Literal(int(row['decreto_anno']), datatype=XSD.integer)))

    if row['stato'] == 'in_gestione' and pd.notna(row['quota_confiscata']):
        g.add((uri_bene, EX_ONT.quotaConfiscata, Literal(float(row['quota_confiscata']), datatype=XSD.decimal)))

print(f'Beni aggiunti al grafo: {len(df):,}')
print(f'Triple totali nel grafo: {len(g):,}')

## 8. Serializzazione e salvataggio

In [ ]:
OUTPUT_TTL = '../ontology/ontologia-beni-confiscati.ttl'
g.serialize(destination=OUTPUT_TTL, format='turtle')
print(f'Grafo salvato in: {OUTPUT_TTL}')

print(f'\nStatistiche grafo:')
print(f'  Triple totali:        {len(g):>10,}')

for cls_name, cls_uri in [('BeneConfiscato', EX_ONT.BeneConfiscato),
                            ('Immobile',       EX_ONT.Immobile),
                            ('Azienda',        EX_ONT.Azienda),
                            ('Comune',         EX_ONT.Comune),
                            ('Provincia',      EX_ONT.Provincia),
                            ('Regione',        EX_ONT.Regione)]:
    n = sum(1 for _ in g.subjects(RDF.type, cls_uri))
    print(f'  {cls_name:<20} {n:>10,}')